In [90]:
import json

DATASET = "FetaQA"
QUESTION_TYPE = Qtype = "answer"
op = "qid_to_output.json"
gold = "qid_to_gold.json"
path1="/home/vivek/harshita/modelling/" + "prompts_gpt4_remaining/" + DATASET + "_" + Qtype + "/" 
path2="/home/vivek/harshita/modelling/" + "prompts_gpt4/" + DATASET + "_" + Qtype + "/"
files_op = [path1+op,path2+op]
files_gold = [path1+gold,path2+gold]

In [91]:
ques_id_to_output = {}
ques_id_to_gold = {}

def merge_json_files(file_paths, output_file):
    merged_data = {}
    for path in file_paths:
        with open(path, 'r') as file:
            data = json.load(file)
            #merged_data.extend(data)
            for i in data:
                merged_data[i] = data[i]
    with open(output_file, 'w') as outfile:
        json.dump(merged_data, outfile)
    #print(len(merged_data))
    return merged_data


In [92]:
ques_id_to_output = merge_json_files(files_op,op)
ques_id_to_gold = merge_json_files(files_gold,gold)


In [93]:
for i,j in ques_id_to_output.items():
    prediction = j.split("Step 2:")[-1].strip()
    print(prediction)
    print(ques_id_to_gold[i])
    print("-----------")


In the 2006-07 season, Rade Prica played for Aalborg BK in the Danish Superliga and scored 19 goals.
Rade Prica scored 19 goals in the Superliga for Aalborg in the 2006-07 season.
-----------
Zachery Ty Bryan was in the film "The Game of Their Lives" made in 2005.
Zachery Ty Bryan appeared as Harry Keough in the 2005 film The Game of Their Lives.
-----------
The series won an award for Outstanding Cinematography for a Single-Camera Series.
Fabian Wagner´s work on the Sherlock episode "A Scandal in Belgravia" and the Game of Thrones episode "Hardhome" earned him two Emmy nominations in 2012 and 2015, respectively.
-----------
The Good Evening Europe Tour began in Hamburg, Germany, on 2 December 2009, and ended in London, England, on 22 December 2009.
The Good Evening Europe Tour began on 2 December 2009, in Hamburg, Germany and concluded on 22 December 2009, at The O2 in London.
-----------
The Kuomintang (KMT) were the winners of the 1996 Taiwan National Assembly, winning 183 seats.
Th

In [94]:
#ques_id_to_output = json.loads(op)
#ques_id_to_gold = json.loads(gold)
TOTAL_LENGTH = len(ques_id_to_gold)
print(TOTAL_LENGTH)

618


In [95]:
import sys
from tqdm import tqdm
sys.path.append("/home/vivek/kunal/evaluation_metrics/")
from exact_match import EvaluationMetrics

if DATASET == "FetaQA":
    ground_truths = []
    predictions = []

    for qid, response in tqdm(ques_id_to_output.items()):
        qid = str(qid)
        gold_ans = ques_id_to_output[qid]
        # TODO: Fix the # after answer in the prompt
        # prompt = response["prompt"][0].split("You must follow the format of answers as demonstrated by the examples above. IMPORTANT: You must give the answer in the format 'Step 2:\n<answer>'.\n\n")[-1]
        # prompt = format_prompt(prompt)
        if type(gold_ans) == list:
            gold_ans = ", ".join([str(_) for _ in gold_ans])

        # print("-----------------")
        
        prediction = response
        if prediction == "Error occurred.":
            prediction = ""
            print("ERROR")

        prediction = prediction.split("Step 2:")[-1].strip()

        print(gold_ans)
        print(prediction)
        print("-----------------")
        ground_truths.append(gold_ans)
        predictions.append(prediction)

    evaluator = EvaluationMetrics()
    bleu_score = evaluator.get_bleu_score(predictions, ground_truths)
    rouge_score = evaluator.get_rouge_score(predictions, ground_truths)
    bleurt_score = evaluator.get_bleurt_score(predictions, ground_truths)

else:

    exact_match = 0
    substring_match = 0
    llm_match = 0
    incorrect = 0
    total = 0
    regex_match_cnt = 0
    f1_scores = []

    llm_evaluations = []
    evaluator = EvaluationMetrics()

    for qid, response in tqdm(ques_id_to_output.items()):

        gold_ans = ques_id_to_gold[qid]
        # TODO: Fix the # after answer in the prompt
        # prompt = response["prompt"][0].split("You must follow the format of answers as demonstrated by the examples above. IMPORTANT: You must give the answer in the format 'Step 2:\n<answer>'.\n\n")[-1]
        # prompt = format_prompt(prompt)
        if type(gold_ans) == list:
            gold_ans = ", ".join([str(_) for _ in gold_ans])

        
        prediction = response
        # print("-----------------")
        # print(gold_ans)
        # print(prediction)
        # print("-----------------")

        if prediction == "Error occurred.": #Temporarily ignoring error responses from gemini
            continue
        prediction = prediction.split("Step 2")[-1].strip()
        if prediction.startswith(":\n"):
            prediction = prediction[2:]

        if prediction.startswith("['"):
            prediction = prediction[2:-2]
        if len(prediction)>0 and prediction[0]=='[':
            prediction = prediction[1:-1]
        ##### GPT EVALUATOR PROMPT CONSTRUCTION #################################################
        # if c<100:
        #  c+=1
        #  gpt_evaluator_prompts[qid] = GPT_EVALUATOR_PROMPT_TEMPLATE.format(question = prompt,answer = gold_ans,candidate=prediction)
        #  print(gpt_evaluator_prompts[qid])

        if evaluator.regex_match(gold_ans, prediction):
            regex_match_cnt+=1

        if evaluator.compute_exact_match(gold_ans, prediction,DATASET):
            exact_match += 1
            substring_match += 1
            llm_match += 1
        else:
            if evaluator.gold_ans_in_prediction(prediction, gold_ans):
                substring_match += 1
                # print("----------------------")
                # print(gold_ans)
                # print(prediction)

            # llm_evaluations.append({
            #     "qid": qid,
            #     "gold_ans": gold_ans,
            #     "prediction": prediction,
            #     "question": test_questions[qid]["question"]
            # })
            else:
                pass
                # print("----------------")
                # print(prompt)
                # print("################")

                # print(gold_ans)
                # print("################")

                # print(prediction)
                # print("################")

                # print(response["prompt"][1])
                # print("----------------")

        f1_scores.append(evaluator.compute_f1_score(gold_ans, prediction))

    mean_f1_score = sum(f1_scores) / TOTAL_LENGTH
    accuracy = exact_match / TOTAL_LENGTH
    substring_accuracy = substring_match / TOTAL_LENGTH
    regex_accuracy = regex_match_cnt / TOTAL_LENGTH

    print(f"{DATASET} {QUESTION_TYPE} Results:")
    print(f"Exact match: {accuracy*100}")
    print(f"Substring match: {substring_accuracy*100}")
    print(f"Mean F1 score: {mean_f1_score}")
    print(f"Regex accuracy: {regex_accuracy}")



100%|██████████| 618/618 [00:00<00:00, 106407.22it/s]


Step 1: To determine which club and division Rade Prica played for in the 2006-07 season and how many goals he scored, we need to look at the row corresponding to the 2006-07 season. The table shows that in the 2006-07 season, Rade Prica played for the club represented by {ENTITY-5} and in the division represented by {ENTITY-6}. The mappings provided indicate that {ENTITY-5} corresponds to Aalborg BK and {ENTITY-6} corresponds to the Danish Superliga. In that season, he scored 19 goals in the league.

Step 2:
In the 2006-07 season, Rade Prica played for Aalborg BK in the Danish Superliga and scored 19 goals.
In the 2006-07 season, Rade Prica played for Aalborg BK in the Danish Superliga and scored 19 goals.
-----------------
Step 1: To determine if Zachery Ty Bryan was in any film made in 2005, we need to look at the "Year" column in the table. The row with the year 2005 shows an image corresponding to {ENTITY-7} in the "Title" column. The image for {ENTITY-7} is the poster for "The Ga

ModuleNotFoundError: No module named 'bleurt'